# Continual learning for NIDS: both experiments in one notebook
Reproduces the paper's experiments with **one engine** (NumPy only, no GPU needed):

| | NSL-KDD (older) | CICIoT2023 (newer, subsample) |
|---|---|---|
| Classes / stages | 5 categories / 4 stages | 34 classes / 5 stages |
| Split | official train/test files | per-seed stratified 70/30 after cleaning |

**Models:** neural network (one hidden layer, 64 ReLU, Adam) and logistic regression.
**Strategies (5, same for both models and both datasets):** fine-tuning, EWC, replay (m = 200; m = 50 and 1000 as ablation), full retraining, class-balanced retraining.
**Seeds:** 10 by default. Statistics: 95% CI, paired t-test and Wilcoxon, per-class precision/recall/F1, samples processed, memory.
Also reproduces the budget-aware test (NSL-KDD).

**Files you need:** `KDDTrain+.txt` and `KDDTest+.txt` (NSL-KDD) and your `ciciot_small.zip` (34 CSV files, first 5,000 rows of each class file).
**Run time (free Colab, 2 CPU cores):** roughly 15-25 minutes in total with 10 seeds (NSL-KDD and CICIoT2023 about 5-10 minutes each, budget test a few minutes); set `SEEDS = 3` in the settings cell for a quick check.
Run the cells from top to bottom.

## 1. Upload the data files

In [ ]:
import os, glob, zipfile, shutil
os.makedirs('NSL_KDD', exist_ok=True); os.makedirs('CICIOT', exist_ok=True)

def have_data():
    nsl = os.path.exists('NSL_KDD/KDDTrain+.txt') and os.path.exists('NSL_KDD/KDDTest+.txt')
    cic = os.path.exists('CICIOT/Benign_Final.csv') and len(glob.glob('CICIOT/*.csv')) >= 34
    return nsl, cic

def sort_uploads(names):
    for f in names:
        low = f.lower()
        if low.endswith('.zip'):
            tmp = 'tmp_zip'; shutil.rmtree(tmp, ignore_errors=True); zipfile.ZipFile(f).extractall(tmp)
            for p in glob.glob(tmp + '/**/*.csv', recursive=True): shutil.copy(p, 'CICIOT/' + os.path.basename(p))
        elif low.endswith('.csv'): shutil.copy(f, 'CICIOT/' + os.path.basename(f))
        elif 'train' in low: shutil.copy(f, 'NSL_KDD/KDDTrain+.txt')
        elif 'test' in low: shutil.copy(f, 'NSL_KDD/KDDTest+.txt')
        else: print('ignored (unknown file):', f)

nsl, cic = have_data()
if not (nsl and cic):
    try:
        from google.colab import files
        print('Choose: KDDTrain+.txt, KDDTest+.txt and your CICIoT zip (you can select all three at once).')
        sort_uploads(list(files.upload().keys()))
    except ImportError:
        sort_uploads(glob.glob('uploads/*'))      # outside Colab: put files in ./uploads
nsl, cic = have_data()
print('NSL-KDD files found:', nsl, '| CICIoT2023 class files found:', len(glob.glob('CICIOT/*.csv')), '(need 34)')
assert nsl and cic, 'a data file is missing - rerun this cell and upload the missing file(s)'

## 2. Write the code files (engine, loaders, budget test)

In [ ]:
%%writefile cl_engine.py
"""Dataset-agnostic class-incremental engine used for BOTH NSL-KDD and CICIoT2023 (NumPy only).
Models: nn (one hidden layer, 64 ReLU, Adam 1e-3) and lr (softmax regression, SGD 0.05, L2 1e-4).
Strategies: finetune, EWC, replay (m=50/200/1000), joint (full retraining), joint_bal (class-balanced retraining).
Classes (CICIoT2023 categories): 0 Benign, 1 DDoS, 2 DoS, 3 Mirai, 4 Recon, 5 Spoofing, 6 Web-based, 7 BruteForce.
Stages (new classes only arrive; large classes first, rare last): S1 {Benign,DDoS} S2 {DoS} S3 {Mirai} S4 {Recon} S5 {Spoofing} S6 {Web-based,BruteForce}."""
import os
for v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'): os.environ[v] = '1'
import sys, json, glob, time, multiprocessing as mp
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support as prfs
import warnings
warnings.filterwarnings('ignore'); np.seterr(all='ignore')

NAMES = ['Benign', 'DDoS', 'DoS', 'Mirai', 'Recon', 'Spoofing', 'Web-based', 'BruteForce']
CIDX = {n: i for i, n in enumerate(NAMES)}
STAGES = [[0, 1], [2], [3], [4], [5], [6, 7]]
K = len(NAMES); EPOCHS, BATCH = 5, 128
G = {}   # arrays shared with workers (inherited through fork)

# ------------------------------------------------------------------ data
def load(npz):
    d = np.load(npz, allow_pickle=True)
    G.update(Xtr=d['Xtr'], ytr=d['ytr'], Xte=d['Xte'], yte=d['yte'])
    return G

# ------------------------------------------------------------------ models (identical to exp3.py)
class Net:
    def __init__(s, kind, rng, din):
        s.kind = kind
        if kind == 'nn':
            s.p = [rng.randn(din, 64).astype(np.float32) * np.sqrt(2.0 / din), np.zeros(64, np.float32),
                   rng.randn(64, K).astype(np.float32) * np.sqrt(1.0 / 64), np.zeros(K, np.float32)]
            s.m = [np.zeros_like(a) for a in s.p]; s.v = [np.zeros_like(a) for a in s.p]; s.t = 0
        else:
            s.p = [np.zeros((din, K), np.float32), np.zeros(K, np.float32)]
    def fwd(s, X):
        if s.kind == 'nn': h = np.maximum(X @ s.p[0] + s.p[1], 0); z = h @ s.p[2] + s.p[3]
        else: h = None; z = X @ s.p[0] + s.p[1]
        z = z - z.max(1, keepdims=True); e = np.exp(z); return h, e / e.sum(1, keepdims=True)
    def grads(s, X, y, w=None):
        h, P = s.fwd(X); n = len(X); dz = P.copy(); dz[np.arange(n), y] -= 1
        if w is not None: dz *= w[:, None]
        dz /= n
        if s.kind == 'nn':
            dh = (dz @ s.p[2].T) * (h > 0); return [X.T @ dh, dh.sum(0), h.T @ dz, dz.sum(0)]
        return [X.T @ dz + 1e-4 * s.p[0], dz.sum(0)]
    def step(s, g):
        if s.kind == 'nn':
            s.t += 1
            for i in range(len(s.p)):
                s.m[i] = 0.9 * s.m[i] + 0.1 * g[i]; s.v[i] = 0.999 * s.v[i] + 0.001 * g[i] ** 2
                mh = s.m[i] / (1 - 0.9 ** s.t); vh = s.v[i] / (1 - 0.999 ** s.t)
                s.p[i] -= 1e-3 * mh / (np.sqrt(vh) + 1e-8)
        else:
            for i in range(len(s.p)): s.p[i] -= 0.05 * g[i]
    def fisher(s, X, y):
        h, P = s.fwd(X); dz = P.copy(); dz[np.arange(len(X)), y] -= 1
        if s.kind == 'nn':
            dh = (dz @ s.p[2].T) * (h > 0)
            F = [(X ** 2).T @ (dh ** 2), (dh ** 2).sum(0), (h ** 2).T @ (dz ** 2), (dz ** 2).sum(0)]
        else:
            F = [(X ** 2).T @ (dz ** 2), (dz ** 2).sum(0)]
        return [f / len(X) for f in F]
    def predict(s, X): return s.fwd(X)[1].argmax(1)

def train(net, idx, rng, w=None, ewc=None):
    Xtr, ytr = G['Xtr'], G['ytr']
    for _ in range(EPOCHS):
        p = idx[rng.permutation(len(idx))]
        for b in range(0, len(p), BATCH):
            bi = p[b:b + BATCH]
            g = net.grads(Xtr[bi], ytr[bi], None if w is None else w[ytr[bi]])
            if ewc is not None and not ewc[3]:
                lam, Fs, anchor, _ = ewc
                g = [gi + lam * Fi * (pi - ai) for gi, Fi, pi, ai in zip(g, Fs, net.p, anchor)]
            net.step(g)
            if ewc is not None and ewc[3]:                  # stable proximal EWC step
                lam, Fs, anchor, _ = ewc; eta = 0.05 if net.kind == 'lr' else 1e-3
                for i in range(len(net.p)):
                    c = eta * lam * Fs[i]; net.p[i] = ((net.p[i] + c * anchor[i]) / (1.0 + c)).astype(np.float32)

def evalfull(net, seen):
    Xte, yte = G['Xte'], G['yte']; m = np.isin(yte, seen); pred = net.predict(Xte[m])
    p, r, f, _ = prfs(yte[m], pred, labels=seen, average=None, zero_division=0)
    return {int(c): dict(p=float(a), r=float(b), f1=float(g)) for c, a, b, g in zip(seen, p, r, f)}

def run(kind, method, seed, m=200, lam=0.0, prox=0):
    Xtr, ytr = G['Xtr'], G['ytr']; din = Xtr.shape[1]
    rng = np.random.RandomState(seed); net = Net(kind, rng, din)
    buf = np.array([], int); seen = []; full = []; samples = 0; Fs = None; anchor = None
    for t, cl in enumerate(STAGES):
        new = np.where(np.isin(ytr, cl))[0]; seen = seen + cl; w = None; ewc = None
        if method in ('joint', 'joint_bal'):
            idx = np.where(np.isin(ytr, seen))[0]; net = Net(kind, rng, din)
            if method == 'joint_bal':
                cnt = np.bincount(ytr[idx], minlength=K).astype(float); w = np.where(cnt > 0, cnt.sum() / (len(seen) * np.maximum(cnt, 1)), 0)
        elif method == 'replay': idx = np.concatenate([new, buf])
        else: idx = new
        if method == 'ewc' and Fs is not None: ewc = (lam, Fs, anchor, prox)
        train(net, idx, rng, w, ewc); samples += len(idx) * EPOCHS
        if method == 'replay':
            buf = np.concatenate([buf] + [rng.choice(np.where(ytr == c)[0], min(m, (ytr == c).sum()), replace=False) for c in cl])
        if method == 'ewc':
            sub = rng.choice(new, min(2000, len(new)), replace=False)
            Fn = net.fisher(Xtr[sub], ytr[sub]); Fs = Fn if Fs is None else [a + b for a, b in zip(Fs, Fn)]
            anchor = [a.copy() for a in net.p]
        full.append(evalfull(net, seen))
    npar = int(sum(a.size for a in net.p))
    return dict(full=full, samples=samples, mem=dict(params=npar, model_bytes=npar * 4, buffer_records=int(len(buf)),
                buffer_bytes=int(len(buf)) * din * 4, ewc_extra_bytes=(2 * npar * 4 if method == 'ewc' else 0)))

CONFIGS = [('finetune', {}), ('replay', dict(m=50)), ('replay', dict(m=200)), ('replay', dict(m=1000)), ('joint', {}), ('joint_bal', {})] + \
          [('ewc', dict(lam=l)) for l in (10., 100., 1e3, 1e4, 1e5)] + [('ewc', dict(lam=l, prox=1)) for l in (10., 100., 1e3, 1e4, 1e5)]
def key(meth, kw): return meth + ''.join(f'_{k}{v:g}' for k, v in kw.items())

def task(args):
    kind, seed = args; t0 = time.time()
    out = {key(m_, kw): run(kind, m_, seed, **kw) for m_, kw in CONFIGS}
    print(f'done {kind} seed {seed} in {time.time()-t0:.0f}s', flush=True)
    return kind, seed, out

def run_all(npz, seeds, procs=4, out_json='cl_results.json'):
    load(npz)
    jobs = [(k, s) for s in range(seeds) for k in ('nn', 'lr')]
    with mp.get_context('fork').Pool(procs) as pool: res = pool.map(task, jobs, chunksize=1)
    out = {'nn': {}, 'lr': {}}
    for kind, seed, o in sorted(res, key=lambda r: (r[0], r[1])):
        for k, v in o.items(): out[kind].setdefault(k, []).append(v)
    json.dump(out, open(out_json, 'w')); return out

# ------------------------------------------------------------------ analysis
def macro(r, t=-1): return float(np.mean([v['f1'] for v in r['full'][t].values()]))
def FB(r):
    T = len(STAGES); R = np.full((T, T), np.nan)
    for t in range(T):
        for i in range(t + 1): R[t, i] = np.mean([r['full'][t][c]['f1'] for c in STAGES[i]])
    return (np.mean([np.nanmax(R[i:T-1, i]) - R[T-1, i] for i in range(T-1)]), np.mean([R[T-1, i] - R[i, i] for i in range(T-1)]))

def analyze(out):
    from scipy import stats
    for kind in ('nn', 'lr'):
        o = {k: [{**r, 'full': [{int(c): v for c, v in st.items()} for st in r['full']]} for r in v] for k, v in out[kind].items()}
        form = (lambda k: k.startswith('ewc') and 'prox1' not in k) if kind == 'nn' else (lambda k: k.startswith('ewc') and 'prox1' in k)
        ewc_keys = [k for k in o if form(k)]
        fin = lambda k: np.array([macro(r) for r in o[k]])
        best = max(ewc_keys, key=lambda k: fin(k).mean())
        print(f'\n===== {kind.upper()}   (EWC form: {"explicit" if kind=="nn" else "proximal"}; strength tuned on test, favours EWC)')
        print('EWC grid:', {k: round(float(fin(k).mean()), 3) for k in ewc_keys})
        rows = [('Fine-tune', 'finetune'), ('EWC (best λ: ' + best + ')', best), ('Replay m=50', 'replay_m50'), ('Replay m=200', 'replay_m200'),
                ('Replay m=1000', 'replay_m1000'), ('Full retraining', 'joint'), ('Retraining, class-balanced', 'joint_bal')]
        for name, k in rows:
            f = fin(k); fb = np.array([FB(r) for r in o[k]]); mem = o[k][0]['mem']; sm = np.mean([r['samples'] for r in o[k]]) / 1e6
            print(f'{name:40s} F1 {f.mean():.3f}±{f.std():.3f}  F {fb[:,0].mean():.3f}  BWT {fb[:,1].mean():.3f}  samples {sm:.2f}M  buf {mem["buffer_records"]} rec ({mem["buffer_bytes"]/1e6:.2f} MB) params {mem["params"]} ewc_extra {mem["ewc_extra_bytes"]/1e3:.0f} KB')
        print('-- paired tests, same seeds: mean diff [95% CI], t p, Wilcoxon p')
        for a, b in [('replay_m200', 'finetune'), ('replay_m200', best), ('replay_m200', 'joint'), ('replay_m1000', 'joint'), ('replay_m200', 'replay_m50'), ('replay_m200', 'joint_bal')]:
            d = fin(a) - fin(b); n = len(d)
            if n < 2: continue
            h = stats.t.ppf(.975, n - 1) * d.std(ddof=1) / np.sqrt(n)
            try: wp = stats.wilcoxon(fin(a), fin(b)).pvalue
            except ValueError: wp = float('nan')
            print(f'  {a} − {b}: {d.mean():+.3f} [{d.mean()-h:+.3f},{d.mean()+h:+.3f}]  t p={stats.ttest_rel(fin(a), fin(b)).pvalue:.4f}  W p={wp:.4f}')
        print('-- per-class final P/R/F1 (mean over seeds)')
        for name, k in [('Fine-tune', 'finetune'), ('EWC', best), ('Replay m=200', 'replay_m200'), ('Full retraining', 'joint'), ('Retraining, class-balanced', 'joint_bal')]:
            s = f'  {name:26s}'
            for c in range(K):
                g = lambda q: np.mean([r['full'][-1][c][q] for r in o[k]])
                s += f' | {NAMES[c]} {g("p"):.2f}/{g("r"):.2f}/{g("f1"):.2f}'
            print(s)

In [ ]:
%%writefile runner.py
"""Loaders and driver for the two datasets. Both use cl_engine (same models, strategies, metrics).
  NSL-KDD   : official train/test files, 5 categories, 4 stages, fixed split (seeds change init, batch order, exemplars).
  CICIoT2023: one CSV per class (first 5,000 rows of each class file), 34 classes, 5 stages, cleaned, per-seed stratified 70/30 split."""
import os, sys, json, time, multiprocessing as mp
for v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'): os.environ[v] = '1'
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
import cl_engine as cg

# ---------------------------------------------------------------- NSL-KDD
def setup_nsl(d):
    cols = [f'f{i}' for i in range(41)] + ['label', 'diff']
    tr = pd.read_csv(os.path.join(d, 'KDDTrain+.txt'), header=None, names=cols)
    te = pd.read_csv(os.path.join(d, 'KDDTest+.txt'), header=None, names=cols)
    dos = 'back land neptune pod smurf teardrop apache2 mailbomb processtable udpstorm worm'.split()
    probe = 'ipsweep nmap portsweep satan mscan saint'.split()
    r2l = 'ftp_write guess_passwd imap multihop phf spy warezclient warezmaster sendmail named snmpgetattack snmpguess xlock xsnoop httptunnel'.split()
    u2r = 'buffer_overflow loadmodule perl rootkit ps sqlattack xterm'.split()
    cat = {'normal': 0}
    for names, c in ((dos, 1), (probe, 2), (r2l, 3), (u2r, 4)): cat.update({a: c for a in names})
    for x in (tr, te): x['y'] = x['label'].map(cat)
    tr, te = tr.dropna(subset=['y']), te.dropna(subset=['y'])
    Xall = pd.get_dummies(pd.concat([tr, te])[[f'f{i}' for i in range(41)]], columns=['f1', 'f2', 'f3']).astype(float)
    Xtr, Xte = Xall.iloc[:len(tr)].values, Xall.iloc[len(tr):].values
    sc = StandardScaler().fit(Xtr)
    cg.G.update(Xtr=sc.transform(Xtr).astype(np.float32), ytr=tr['y'].values.astype(int),
                Xte=sc.transform(Xte).astype(np.float32), yte=te['y'].values.astype(int))
    cg.NAMES = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']; cg.K = 5; cg.STAGES = [[0, 1], [2], [3], [4]]
    print('NSL-KDD train', np.bincount(cg.G['ytr']).tolist(), 'test', np.bincount(cg.G['yte']).tolist(), 'features', Xtr.shape[1])

# ---------------------------------------------------------------- CICIoT2023
STAGE_NAMES = [
 ['Benign_Final','DDoS-ICMP_Flood','DDoS-UDP_Flood','DDoS-SYN_Flood','DDoS-TCP_Flood'],
 ['DDoS-ACK_Fragmentation','DDoS-HTTP_Flood','DDoS-ICMP_Fragmentation','DDoS-PSHACK_FLOOD','DDoS-RSTFINFLOOD','DDoS-SlowLoris','DDoS-SynonymousIP_Flood','DDoS-UDP_Fragmentation'],
 ['DoS-HTTP_Flood','DoS-SYN_Flood','DoS-TCP_Flood','DoS-UDP_Flood','Mirai-greeth_flood','Mirai-greip_flood','Mirai-udpplain'],
 ['Recon-HostDiscovery','Recon-OSScan','Recon-PingSweep','Recon-PortScan','VulnerabilityScan','MITM-ArpSpoofing','DNS_Spoofing'],
 ['BrowserHijacking','CommandInjection','DictionaryBruteForce','SqlInjection','Uploading_Attack','XSS','Backdoor_Malware']]
DX = DY = None

def setup_ciciot(d):
    global DX, DY
    names = [n for s in STAGE_NAMES for n in s]
    cg.NAMES = names; cg.K = len(names); cg.STAGES = [[names.index(n) for n in s] for s in STAGE_NAMES]
    X, y = [], []
    for c, n in enumerate(names):
        df = pd.read_csv(os.path.join(d, n + '.csv')).replace([np.inf, -np.inf], np.nan).dropna()
        X.append(df.values.astype(np.float32)); y.append(np.full(len(df), c))
    X = np.concatenate(X); y = np.concatenate(y); n0 = len(X)
    h = pd.util.hash_pandas_object(pd.DataFrame(X), index=False).values
    nlab = pd.Series(y).groupby(h).transform('nunique').values
    conflict = nlab > 1; dup = pd.Series(h).duplicated().values & ~conflict; keep = ~conflict & ~dup
    print(f'CICIoT2023 rows {n0} | ambiguous removed {int(conflict.sum())} | exact duplicates removed {int(dup.sum())} | kept {int(keep.sum())}')
    DX, DY = X[keep], y[keep]; print('class counts', np.bincount(DY).tolist())

def split_ciciot(seed, test=0.3):
    rng = np.random.RandomState(1000 + seed); tr, te = [], []
    for c in range(cg.K):
        i = np.where(DY == c)[0]; p = rng.permutation(len(i)); nte = int(round(test * len(i)))
        te.append(i[p[:nte]]); tr.append(i[p[nte:]])
    tr, te = np.concatenate(tr), np.concatenate(te)
    mu = DX[tr].mean(0); sd = DX[tr].std(0) + 1e-6
    f = lambda A: np.clip((A - mu) / sd, -10, 10).astype(np.float32)
    cg.G.update(Xtr=f(DX[tr]), ytr=DY[tr], Xte=f(DX[te]), yte=DY[te])

# ---------------------------------------------------------------- driver
CFG = {'nn': [c for c in cg.CONFIGS if not (c[0] == 'ewc' and c[1].get('prox'))],       # explicit EWC for the network
       'lr': [c for c in cg.CONFIGS if not (c[0] == 'ewc' and not c[1].get('prox'))]}   # proximal EWC for logistic regression
DATASET = None
def _task(a):
    kind, seed = a; t0 = time.time()
    if DATASET == 'ciciot': split_ciciot(seed)
    out = {cg.key(m, kw): cg.run(kind, m, seed, **kw) for m, kw in CFG[kind]}
    print(f'  done {DATASET} {kind} seed {seed} in {time.time()-t0:.0f}s', flush=True); return kind, seed, out

def run_dataset(name, data_dir, seeds=10, procs=2, out_json=None):
    global DATASET
    DATASET = name
    (setup_nsl if name == 'nsl' else setup_ciciot)(data_dir)
    jobs = [(k, s) for s in range(seeds) for k in ('nn', 'lr')]
    with mp.get_context('fork').Pool(procs) as pool: res = pool.map(_task, jobs, chunksize=1)
    out = {'nn': {}, 'lr': {}}
    for kind, seed, o in sorted(res, key=lambda r: (r[0], r[1])):
        for k, v in o.items(): out[kind].setdefault(k, []).append(v)
    out_json = out_json or f'results_{name}_{seeds}seeds.json'
    json.dump(out, open(out_json, 'w')); print('saved', out_json); return out

def set_globals(name):
    """Point the engine at one dataset's class names/stages (needed before analysis when both datasets were run)."""
    if name == 'nsl':
        cg.NAMES = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']; cg.K = 5; cg.STAGES = [[0, 1], [2], [3], [4]]
    else:
        names = [n for s in STAGE_NAMES for n in s]
        cg.NAMES = names; cg.K = len(names); cg.STAGES = [[names.index(n) for n in s] for s in STAGE_NAMES]

In [ ]:
%%writefile budget.py
"""Budget-aware test (Eq. 3) on NSL-KDD. Actions at stages 2-4: replay (m=200) or retrain from scratch.
Budget 1.5M samples; rule: after a tentative replay update, if the validation macro-F1 on OLD classes drops by more than 0.10
and the remaining budget covers a retraining, discard the update and retrain. Validation = 10% of each class's training records
(never trained on, never the test file)."""
import json, multiprocessing as mp
import numpy as np
from sklearn.metrics import f1_score
import cl_engine as cg
M = 200
CFGB = dict(dataset='nsl', budget=1.5e6, taus=(0.10,))     # set by run_budget
def policies():
    T = len(cg.STAGES)
    return ['replay'] + [f'retrain_at_{k}' for k in range(2, T + 1)] + ['retrain_always'] + [f'budget_t{t:g}' for t in CFGB['taus']]

def _run(kind, policy, seed, val, trmask):
    Xtr, ytr = cg.G['Xtr'], cg.G['ytr']; din = Xtr.shape[1]
    def vf1(net, seen):
        m = np.isin(ytr[val], seen)
        return float(np.mean(f1_score(ytr[val][m], net.predict(Xtr[val][m]), labels=seen, average=None, zero_division=0)))
    rng = np.random.RandomState(seed); net = cg.Net(kind, rng, din)
    buf = np.array([], int); seen = []; samples = 0; log = []
    for t, cl in enumerate(cg.STAGES):
        new = np.where(np.isin(ytr, cl) & trmask)[0]; old = list(seen); seen = seen + cl
        all_idx = np.where(np.isin(ytr, seen) & trmask)[0]
        if t == 0:
            cg.train(net, new, rng); samples += len(new) * cg.EPOCHS; act = 'train'
        else:
            cost = len(all_idx) * cg.EPOCHS
            if policy.startswith('budget'):
                before = vf1(net, old); idx = np.concatenate([new, buf]); cg.train(net, idx, rng); samples += len(idx) * cg.EPOCHS
                drop = before - vf1(net, old)
                if drop > float(policy.split('_t')[1]) and samples + cost <= CFGB['budget']:
                    net = cg.Net(kind, rng, din); cg.train(net, all_idx, rng); samples += cost; act = 'retrain'
                else: act = 'replay'
            else:
                do = (policy == 'retrain_always') or (policy.startswith('retrain_at_') and t + 1 == int(policy.split('_')[-1]))
                if do: net = cg.Net(kind, rng, din); cg.train(net, all_idx, rng); samples += cost; act = 'retrain'
                else:
                    idx = np.concatenate([new, buf]); cg.train(net, idx, rng); samples += len(idx) * cg.EPOCHS; act = 'replay'
        buf = np.concatenate([buf] + [rng.choice(np.where((ytr == c) & trmask)[0], min(M, ((ytr == c) & trmask).sum()), replace=False) for c in cl])
        log.append(act)
    pc = cg.evalfull(net, seen)
    return dict(final=float(np.mean([v['f1'] for v in pc.values()])), samples=samples, actions=log)

def _task(a):
    import runner
    kind, seed = a
    if CFGB['dataset'] == 'ciciot': runner.split_ciciot(seed)
    ytr = cg.G['ytr']; vrng = np.random.RandomState(123)
    val = np.concatenate([vrng.choice(np.where(ytr == c)[0], max(1, int(0.1 * (ytr == c).sum())), replace=False) for c in range(cg.K)])
    trmask = np.ones(len(ytr), bool); trmask[val] = False
    return kind, seed, {p: _run(kind, p, seed, val, trmask) for p in policies()}

def run_budget(dataset='nsl', seeds=10, procs=2, budget=1.5e6, taus=(0.10,), out_json='results_budget.json'):
    CFGB.update(dataset=dataset, budget=budget, taus=tuple(taus))
    jobs = [(k, s) for s in range(seeds) for k in ('nn', 'lr')]
    with mp.get_context('fork').Pool(procs) as pool: res = pool.map(_task, jobs, chunksize=1)
    out = {'nn': {p: [] for p in policies()}, 'lr': {p: [] for p in policies()}}
    for kind, seed, o in sorted(res, key=lambda r: (r[0], r[1])):
        for p, v in o.items(): out[kind][p].append(v)
    json.dump(out, open(out_json, 'w')); return out

def report(out):
    from scipy import stats
    for kind in ('nn', 'lr'):
        print(f'\n== {kind.upper()} budget test (macro-F1 after the last stage, mean±sd over seeds)')
        base = np.array([r['final'] for r in out[kind]['replay']])
        for p in out[kind]:
            f = np.array([r['final'] for r in out[kind][p]]); sm = np.mean([r['samples'] for r in out[kind][p]]) / 1e6
            ex = ''
            if p != 'replay' and len(f) > 1 and np.any(f != base): ex = f'  vs replay {np.mean(f-base):+.3f} (t p={stats.ttest_rel(f, base).pvalue:.3f}, W p={stats.wilcoxon(f, base).pvalue:.4f})'
            print(f'  {p:15s} {f.mean():.3f}±{f.std(ddof=1) if len(f)>1 else 0:.3f}  {sm:.2f}M samples{ex}')
        for p in [q for q in out[kind] if q.startswith('budget')]:
            fired = sum(a == 'retrain' for r in out[kind][p] for a in r['actions']); n = len(out[kind][p])
            print(f'  {p}: rule fired {fired} times in {n} runs ({sum(any(a=="retrain" for a in r["actions"]) for r in out[kind][p])} runs at least once)')

## 3. Settings

In [ ]:
import os, sys, time
sys.path.insert(0, os.getcwd())
SEEDS = 10                       # 10 for the paper; 3 for a quick check
PROCS = min(os.cpu_count() or 2, 4)
print('seeds', SEEDS, '| processes', PROCS)

## 4. NSL-KDD: 2 models x 5 strategies (+ replay m = 50/1000 ablation)

In [ ]:
import runner, cl_engine as cg
t0 = time.time()
out_nsl = runner.run_dataset('nsl', 'NSL_KDD', seeds=SEEDS, procs=PROCS, out_json='results_nslkdd.json')
print(f'NSL-KDD finished in {(time.time()-t0)/60:.1f} min')

## 5. CICIoT2023: same models and strategies

In [ ]:
t0 = time.time()
out_cic = runner.run_dataset('ciciot', 'CICIOT', seeds=SEEDS, procs=PROCS, out_json='results_ciciot2023.json')
print(f'CICIoT2023 finished in {(time.time()-t0)/60:.1f} min')

## 6. Budget-aware test (both datasets)
Rule written before the runs: retrain from scratch if the validation macro-F1 on old classes drops by more than tau and the budget allows (NSL-KDD budget 1.5M samples, CICIoT2023 1.2M). tau = 0.10 is the pre-specified value; the other values are a sensitivity check.

In [ ]:
import budget
t0 = time.time()
runner.setup_nsl('NSL_KDD'); runner.DATASET = 'nsl'
out_bud_nsl = budget.run_budget('nsl', seeds=SEEDS, procs=PROCS, budget=1.5e6, taus=(0.02, 0.05, 0.10), out_json='results_budget_nslkdd.json')
runner.setup_ciciot('CICIOT'); runner.DATASET = 'ciciot'; runner.set_globals('ciciot')
out_bud_cic = budget.run_budget('ciciot', seeds=SEEDS, procs=PROCS, budget=1.2e6, taus=(0.05, 0.10, 0.20), out_json='results_budget_ciciot2023.json')
print(f'budget tests finished in {(time.time()-t0)/60:.1f} min')

## 7. Results: NSL-KDD (macro-F1, forgetting F, samples, memory, paired tests, per-class P/R/F1)

In [ ]:
runner.set_globals('nsl'); cg.analyze(out_nsl)

## 8. Results: CICIoT2023

In [ ]:
runner.set_globals('ciciot'); cg.analyze(out_cic)

## 9. Budget-aware test results

In [ ]:
print('########## NSL-KDD'); budget.CFGB.update(dataset='nsl', taus=(0.02, 0.05, 0.10)); budget.report(out_bud_nsl)
print('\n########## CICIoT2023'); budget.CFGB.update(dataset='ciciot', taus=(0.05, 0.10, 0.20)); budget.report(out_bud_cic)

## 10. Download everything

In [ ]:
files_to_zip = ['results_nslkdd.json', 'results_ciciot2023.json', 'results_budget_nslkdd.json', 'results_budget_ciciot2023.json', 'cl_engine.py', 'runner.py', 'budget.py']
import zipfile
with zipfile.ZipFile('cl_nids_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files_to_zip: z.write(f)
print('saved cl_nids_results.zip')
try:
    from google.colab import files; files.download('cl_nids_results.zip')
except ImportError: pass